In [23]:
import sqlite3
import pandas as pd
import polars as pl
pd.options.mode.chained_assignment = None

import sys
sys.path.append('../')
sys.path.append('../networks')

from functions.env import  DB_SCIENCE_PATH_NEW, GRAPH_RESULTS

conn = sqlite3.connect(DB_SCIENCE_PATH_NEW)

from functions.datamodel import OptimumParameter
from functions.feat_network import get_edge_node_table
from functions.feat_visualization import sygma_graph_leiden


# compute average distance:
import pickle
from tqdm import tqdm
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import warnings
# Use the filterwarnings function to filter and suppress FutureWarning
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
# Function to calculate cosine similarity between position vectors
def cosine_similarity(vector1, vector2):
    dot_product = np.dot(vector1, vector2)
    norm1 = np.linalg.norm(vector1)
    norm2 = np.linalg.norm(vector2)
    return dot_product / (norm1 * norm2)

In [24]:
import glob

list_df = []
paths = glob.glob('../networks/data/weighted/*')
for path in paths:
    df = pd.read_csv(path, index_col = [0])
    df = df[['wikidata_id', 'meta_occupation']]
    df['meta_occupation'] = df['meta_occupation'].apply(lambda x: x.split(' | '))
    df = df.explode('meta_occupation')
    df = df.reset_index(drop=True)
    df.columns = ['source', 'target']
    df['weight'] = 1
    list_df.append(df)

In [25]:
def data_to_edge(df_test):

    source_to_targets = df_test.groupby('source')['target'].apply(list).to_dict()

    # Step 2: Create a dictionary to count co-occurrences
    co_occurrences = {}

    # Iterate through each source
    for source, targets in source_to_targets.items():
        # Count each pair of targets that appear together
        for i in range(len(targets)):
            for j in range(i+1, len(targets)):
                target_pair = tuple(sorted([targets[i], targets[j]]))
                if target_pair in co_occurrences:
                    co_occurrences[target_pair] += 1
                else:
                    co_occurrences[target_pair] = 1

    # Step 3: Convert co-occurrences to a dataframe
    co_occur_df = pd.DataFrame([
        {'source': pair[0], 'target': pair[1], 'weight': count}
        for pair, count in co_occurrences.items()
    ])

    # Step 4: Sort the dataframe by weight in descending order
    co_occur_df = co_occur_df.sort_values('weight', ascending=False)
    co_occur_df = co_occur_df[co_occur_df['weight']>=3]

    return co_occur_df

In [26]:
list_edge_filtered = []
list_nodes = []
for element in tqdm(list_df):
    df_net = pl.from_pandas(element)
    df_net = df_net.to_pandas()

    df_cooc = data_to_edge(df_net)

    # df_edge, df_nodes = get_edge_node_table(df_net)
    df_cooc = df_cooc[
        df_cooc["source"] != df_cooc["target"]
    ]


    df_nodes = df_net.groupby(["target"])["weight"].sum().reset_index()
    df_nodes.columns = ["node", "sum_weight"]
    df_nodes = df_nodes.sort_values("sum_weight", ascending=False)
    df_nodes = df_nodes.reset_index(drop=True)


    # df_edge_filter = df_edge_filter[
    #     df_edge_filter["rank_count"] <= 3
    # ]
    df_edge_filter = df_cooc[['source', 'target', 'weight']].copy()
    #maxtrix = df_edge_filter.pivot(index='source', columns='target', values='weight').fillna(0)
    
    list_edge_filtered.append(df_edge_filter)
    list_nodes.append(df_nodes)

100%|██████████| 100/100 [00:05<00:00, 17.21it/s]


In [27]:
df_edge_filtered_average = pd.concat([x for x in list_edge_filtered])
df_edge_filtered_average = df_edge_filtered_average.groupby(['source', 'target'])['weight'].sum().reset_index()
df_edge_filtered_average['weight'] = df_edge_filtered_average['weight'] / 100 # divide by the number of dataset to get the mean
df_edge_filtered_average = df_edge_filtered_average.sort_values('weight', ascending=False)
df_edge_filtered_average = df_edge_filtered_average.reset_index(drop=True)
df_edge_filtered_average.to_csv('edges_list_filtered/average_edges_list_global_thresh.csv')

In [28]:
df_nodes_average = pd.concat([x for x in list_nodes])
df_nodes_average = df_nodes_average.groupby(['node'])['sum_weight'].mean().reset_index()

In [29]:
# visualization
df_partition, g = sygma_graph_leiden(
df_edge_filtered_average,
df_nodes_average,
edge_bins=10,
node_bins=10,
filepath='final_graph/average_network_global_thresh_pre_1700.html')